# Programme 2 - `autoReadForm` (assemblage & MAIN)
Implementation des fonctions du **cahier des charges 3.4**. Pour chaque PDF d'un dossier, on ouvre le PDF (1 image/page), on recale la page 1, on lit la grille (STUDENT ID, GROUP), les conditions, la signature, le cryptogramme, puis les pages de questions (QCM), et on ecrit `EXAM_FORMXX_abcd.xlsx` (onglets **PAGE-01** et **EXAM**) dans `EXAM_FORMXX_RESULTS`.

**Consigne 4.1 respectee** : tout le graphique (cases, grilles, coins, cryptogramme) est traite en **bas niveau** (Sobel, Otsu, morphologie, homographie, `regionprops`, densite d'encre, NCC). Les textes imprimes/manuscrits passent par des **hooks** ML (OCR/CNN) qui renvoient une valeur neutre tant qu'ils ne sont pas branches, afin que le main s'execute **sans planter**.

## 1. Primitives & fonctions du cahier des charges

In [ ]:
# ============================================================================
#  PROGRAMME 2 - Lecture automatique des formulaires (assemblage)
#  Fonctions du cahier des charges 3.4 : autoReadForm / autoReadFormID
#  Toutes les primitives bas niveau sont dans reader_pdf.py (module commente).
# ============================================================================
import os, glob, numpy as np
import reader_pdf as R
from openpyxl import Workbook

# ---- PARAMETRES centralises (consigne 8 : pas de valeurs codees en dur) -----
PARAMS = dict(dpi=200, crypto_thr=0.60)

# ROI en FRACTIONS de la page NORMALISEE 1654x2339 (formulaire identique FORM1/2/3).
# -> a ajuster avec la cellule DEBUG plus bas si le gabarit change.
ROI = {
    "student_id": (0.81, 0.99, 0.18, 0.45, 5),   # x0,x1,y0,y1,n_chiffres
    "group":      (0.49, 0.70, 0.185, 0.45),
    "cond_band":  (0.60, 0.645),                 # bande des 5 paires YES/NO
    "qcm_strip":  (0.04, 0.12),                  # colonne de gauche (cases QCM)
    "crypto":     (0.235, 0.915, 0.07, 0.06),     # x,y,w,h (image BRUTE)
    "signature":  (0.02, 0.45, 0.27, 0.46),
}
LABELS_P1 = ["Module","Professor","Date","Code","Notes de cours","Notes manuscrites",
    "Ordinateur portable","Calculatrice","Feuilles brouillon","Note maximale","Note pour valider",
    "","Prenom","Nom","Validation signature","Group","STUDENT ID","Validation cryptogramme"]

# ---- lectures specifiques --------------------------------------------------
def read_conditions(g):
    H,W=g.shape; y0,y1=ROI["cond_band"]; reg=g[int(y0*H):int(y1*H),:]
    bx=sorted(R.detect_boxes(reg,size=3,mn=30,mind=10,dens=0.6),key=lambda b:b[0])
    if len(bx)<10: return [None]*5
    inks=[R.interior_ink(reg,b[2],sh=0.2,t=0.65) for b in bx[:10]]
    return [1 if inks[i]>=inks[i+1] else 0 for i in range(0,10,2)]

def validate_signature(g):
    H,W=g.shape; x0,x1,y0,y1=ROI["signature"]
    reg=g[int(y0*H):int(y1*H),int(x0*W):int(x1*W)]
    return 1 if (reg<0.5).mean()>0.01 else 0

# Hooks ML fail-soft : textes imprimes / manuscrits = haut niveau autorise (4.1).
# Renvoient "" pour ne jamais faire planter le main ; a brancher (OCR / CNN).
def read_printed(g, field):     return ""
def read_handwriting(g, field): return ""

# ---- autoReadFormID : 1 PDF -> 1 fichier .xlsx -----------------------------
def autoReadFormID(pdf_path, signatures_dir, results_dir):
    pages = R.pdf_to_pages(pdf_path, dpi=PARAMS["dpi"])
    norms = [R.norm_gray(p) for p in pages]
    g1    = norms[0]

    sid    = R.read_digit_grid(g1, *ROI["student_id"])
    group  = R.read_group_grid(g1, *ROI["group"])
    cond   = read_conditions(g1)
    sigval = validate_signature(g1)
    crypto = R.cryptograms_identical([R.crypto_sig(p, ROI["crypto"]) for p in pages],
                                     thr=PARAMS["crypto_thr"])

    # ----- onglet PAGE-01 -----
    vals = [read_printed(g1,"module"), read_printed(g1,"prof"),
            read_printed(g1,"date"), read_printed(g1,"code"),
            cond[0],cond[1],cond[2],cond[3],cond[4],
            read_printed(g1,"note_max"), read_printed(g1,"note_valid"),
            "", read_handwriting(g1,"prenom"), read_handwriting(g1,"nom"),
            sigval, group, sid, crypto]
    wb=Workbook(); ws=wb.active; ws.title="PAGE-01"
    for i,(lab,v) in enumerate(zip(LABELS_P1, vals), start=1):
        ws.cell(i,1,lab); ws.cell(i,2, v if v!="" else None)

    # ----- onglet EXAM (pages 5 -> fin) -----
    ws2=wb.create_sheet("EXAM")
    ws2.append(["QUESTION"]+[f"CHOIX {c}" for c in "ABCDEFGH"]+["MANTISSE","EXPOSANT","UNITE"])
    qnum=0
    for idx in range(4, len(pages)):
        for blk in R.read_exam_page(pages[idx], x0f=ROI["qcm_strip"][0], x1f=ROI["qcm_strip"][1]):
            qnum+=1; row=[qnum]+[""]*8+["","",""]
            if blk["type"]=="QCM":
                for ch in blk["choices"]:
                    row[1+"ABCDEFGH".index(ch)] = 1
            else:
                row[9]=read_handwriting(None,"mantisse")
                row[10]=read_handwriting(None,"exposant")
                row[11]=read_handwriting(None,"unite")
            ws2.append(row)

    os.makedirs(results_dir, exist_ok=True)
    out=os.path.join(results_dir, os.path.splitext(os.path.basename(pdf_path))[0]+".xlsx")
    wb.save(out)
    return out, dict(sid=sid, group=group, cond=cond, sig=sigval, crypto=crypto, nq=qnum)

# ---- autoReadForm : tout un dossier ---------------------------------------
def autoReadForm(pdf_dir, signatures_dir, results_dir):
    res={}
    for pdf in sorted(glob.glob(os.path.join(pdf_dir, "*.pdf"))):
        try:
            out, info = autoReadFormID(pdf, signatures_dir, results_dir)
            print(f"  OK {os.path.basename(pdf)} -> {os.path.basename(out)} | "
                  f"ID={info['sid']} GROUP={info['group']} cond={info['cond']} "
                  f"sig={info['sig']} crypto={info['crypto']} nQ={info['nq']}")
            res[pdf]=info
        except Exception as e:                     # fail-soft : un PDF KO n'arrete pas le lot
            print(f"  ERREUR {os.path.basename(pdf)} : {e}"); res[pdf]={"ERROR":str(e)}
    return res


## 2. MAIN - boucle sur FORM1 / FORM2 / FORM3

In [ ]:
# ============================  MAIN  ========================================
# Traite FORM1, FORM2, FORM3 ; ecrit un .xlsx par PDF dans <dossier>_RESULTS.
STUDENT_CLASS_SIGNATURES = "SIGNATURES"     # base des signatures (Programme 1)

for dossier in ["FORM1", "FORM2", "FORM3"]:
    if not os.path.isdir(dossier):
        print(f"[skip] dossier absent : {dossier}"); continue
    results = dossier + "_RESULTS"
    print(f"\n=== {dossier} -> {results} ===")
    autoReadForm(dossier, STUDENT_CLASS_SIGNATURES, results)
print("\nTermine.")


## 3. (option) Verifier un fichier genere

In [ ]:
from openpyxl import load_workbook
import glob
xs = sorted(glob.glob(os.path.join("FORM1_RESULTS","*.xlsx")))
if xs:
    wb=load_workbook(xs[0]); print("Fichier :", xs[0], "| onglets :", wb.sheetnames)
    print("\n[PAGE-01]")
    for r in wb["PAGE-01"].iter_rows(values_only=True):
        if any(v is not None for v in r): print("  ", r)
    print("\n[EXAM]")
    for r in wb["EXAM"].iter_rows(values_only=True): print("  ", r)


## 4. DEBUG - regler les ROI

In [ ]:
# ===================  DEBUG : visualiser les ROI  ==========================
# Superpose les zones lues sur la page 1 normalisee pour regler les ROI au besoin.
import matplotlib.pyplot as plt, matplotlib.patches as patches, glob, os
pdfs = sorted(glob.glob(os.path.join("FORM1","*.pdf")))
if pdfs:
    pages = R.pdf_to_pages(pdfs[0], dpi=PARAMS["dpi"]); g = R.norm_gray(pages[0]); H,W = g.shape
    fig,ax = plt.subplots(figsize=(7,10)); ax.imshow(g, cmap='gray')
    def box(x0,x1,y0,y1,c,lab):
        ax.add_patch(patches.Rectangle((x0*W,y0*H),(x1-x0)*W,(y1-y0)*H,fill=False,ec=c,lw=2))
        ax.text(x0*W, y0*H-6, lab, color=c, fontsize=9)
    s=ROI["student_id"]; box(s[0],s[1],s[2],s[3],'r','STUDENT ID')
    gp=ROI["group"];     box(gp[0],gp[1],gp[2],gp[3],'lime','GROUP')
    box(0.0,1.0,ROI["cond_band"][0],ROI["cond_band"][1],'cyan','CONDITIONS')
    si=ROI["signature"]; box(si[0],si[1],si[2],si[3],'orange','SIGNATURE')
    cr=ROI["crypto"];    box(cr[0],cr[0]+cr[2],cr[1],cr[1]+cr[3],'magenta','CRYPTO')
    ax.set_title("ROI page 1 (rouge=ID, vert=group, cyan=conditions, orange=signature, magenta=crypto)")
    ax.axis('off'); plt.show()
else:
    print("Place des PDF dans FORM1/ pour visualiser les ROI.")
